In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, recall_score, f1_score

# 1. Load Data
df = pd.read_csv('demand_agent_training.csv')
df['week_start_date'] = pd.to_datetime(df['week_start_date'])

# 2. Feature Selection & Categorical Encoding
target_col = 'next_week_demand_target_units'
feature_cols = [
    'year', 'week_number', 'demand_rank', 'weekday_units_sold', 'weekend_units_sold', 
    'units_sold', 'avg_selling_price_rs', 'promotion_flag', 'discount_pct', 'holiday_flag', 
    'temperature_c', 'rainfall_mm', 'humidity_pct', 'stockout_flag',
    'location_id', 'product_id', 'category_name', 'brand', 'season', 'weather_condition'
]

cat_cols = ['location_id', 'product_id', 'category_name', 'brand', 'season', 'weather_condition']

data = df[feature_cols + [target_col, 'week_start_date']].copy()
for col in cat_cols:
    data[col] = data[col].astype('category')

# 3. Chronological Time-Series Split (80% Train, 20% Test)
unique_dates = sorted(data['week_start_date'].unique())
train_cutoff = unique_dates[int(len(unique_dates) * 0.8)]

train_data = data[data['week_start_date'] <= train_cutoff]
test_data = data[data['week_start_date'] > train_cutoff]

X_train, y_train = train_data[feature_cols], train_data[target_col]
X_test, y_test = test_data[feature_cols], test_data[target_col]

# 4. Fit Model (Regressor)
model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    random_state=42
)
model.fit(X_train, y_train)

# 5. Predict & Evaluate Regression Metrics (RMSE, MAE, R²)
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Regression Metrics ---")
print(f"Test RMSE: {rmse:.2f}")
print(f"Test MAE:  {mae:.2f}")
print(f"Test R²:   {r2:.4f}")

# 6. Evaluate Classification Metrics via Thresholding (Accuracy, Recall, F1-Score)
# Convert continuous values into binary classes (1 if above median demand, 0 otherwise)
threshold = y_train.median()
y_test_binary = (y_test > threshold).astype(int)
y_pred_binary = (y_pred > threshold).astype(int)

accuracy = accuracy_score(y_test_binary, y_pred_binary)
recall = recall_score(y_test_binary, y_pred_binary)
f1 = f1_score(y_test_binary, y_pred_binary)

print("\n--- Classification Metrics (Thresholded at Median Demand) ---")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Recall:   {recall:.4f}")
print(f"Test F1-Score: {f1:.4f}")

--- Regression Metrics ---
Test RMSE: 20.06
Test MAE:  14.50
Test R²:   0.7925

--- Classification Metrics (Thresholded at Median Demand) ---
Test Accuracy: 0.8672
Test Recall:   0.8625
Test F1-Score: 0.8707
